# Desafio 1 — Pipeline Bronze → Silver → Gold: Taxas de Câmbio (Frankfurter API)

**Conceitos praticados:** ingestão batch incremental, Delta append-only na Bronze, `MERGE INTO` idempotente por chave composta, window functions, `OPTIMIZE + ZORDER`.

**Seções do exame cobertas:** S2 (Data Ingestion) e S3 (Data Transformation and Modeling)

**API:** https://api.frankfurter.app — 100% pública, sem chave de API.

> ⚠️ Se o outbound internet estiver bloqueado na Free Edition, use o **Plano B** documentado na Célula 7.

## Passo 0 — Setup: criar o schema `exchange` (execute uma única vez)

In [ ]:
%sql
-- Célula 3 — Setup: criação do schema exchange
CREATE SCHEMA IF NOT EXISTS workshop_weather.exchange
  COMMENT 'Taxas de câmbio — Frankfurter e ExchangeRate-API';

## Passo 1 — Bronze: ingestão do payload bruto

Princípio Bronze: gravar o dado **exatamente como veio da fonte**, sem transformações.
O único enriquecimento permitido são metadados de controle de ingestão.

In [ ]:
# Célula 5 — Imports e configuração
import requests
import json
import uuid
from datetime import date, datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructType, StructField, TimestampType

CATALOG = "workshop_weather"
SCHEMA  = "exchange"
TABLE_BRONZE = f"{CATALOG}.{SCHEMA}.raw_frankfurter"

# Moedas de interesse
TARGET_CURRENCIES = "BRL,EUR,GBP,JPY,ARS"
BASE_CURRENCY = "USD"

print(f"Tabela Bronze alvo: {TABLE_BRONZE}")

In [ ]:
# Célula 6 — Função de extração com tratamento de erro
def fetch_frankfurter(url: str) -> dict | None:
    """
    Faz a requisição HTTP para a Frankfurter API.
    Retorna o dicionário JSON ou None em caso de falha.
    Nunca levanta exceção — falhas são logadas e o pipeline continua.
    """
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"[ERRO] Falha ao acessar {url}: {e}")
        return None

# Teste rápido — valida conectividade antes de prosseguir
url_latest = f"https://api.frankfurter.app/latest?from={BASE_CURRENCY}&to={TARGET_CURRENCIES}"
test = fetch_frankfurter(url_latest)

if test:
    print(f"[OK] Conectividade confirmada. Data da taxa: {test.get('date')}")
    print(f"     Moedas retornadas: {list(test.get('rates', {}).keys())}")
else:
    print("[AVISO] API inacessível. Siga para o Plano B (próxima célula).")

### ⚠️ Plano B — API inacessível (outbound bloqueado)

Execute o script abaixo **localmente** na sua máquina, faça upload do arquivo gerado para o Volume e ajuste `USE_PLAN_B = True` na Célula 8.

```python
# Execute LOCALMENTE (não no Databricks):
import requests, json

url = "https://api.frankfurter.app/2024-01-01..2024-12-31?from=USD&to=BRL,EUR,GBP,JPY,ARS"
response = requests.get(url)
with open("frankfurter_2024.json", "w") as f:
    json.dump(response.json(), f)
print(f"Arquivo salvo: frankfurter_2024.json")
```

Depois: **Data > Volumes > workshop_weather > bronze > landing** > crie pasta `frankfurter` > faça upload do arquivo.

In [ ]:
# Célula 8 — Flag Plano B + carga histórica 2024
# ============================================================
# CONFIGURAÇÃO: defina True se estiver usando o Plano B
USE_PLAN_B = False
PLAN_B_PATH = "/Volumes/workshop_weather/bronze/landing/frankfurter/frankfurter_2024.json"
# ============================================================

batch_id = str(uuid.uuid4())
ingestion_ts = datetime.utcnow()

if not USE_PLAN_B:
    # Carga histórica via API — série temporal de 2024 completa
    url_history = (
        f"https://api.frankfurter.app/2024-01-01..2024-12-31"
        f"?from={BASE_CURRENCY}&to={TARGET_CURRENCIES}"
    )
    print(f"[INFO] Buscando série histórica 2024...")
    json_data = fetch_frankfurter(url_history)

    if json_data is None:
        raise RuntimeError("API falhou. Ative USE_PLAN_B = True e siga as instruções do Plano B.")

    # A série temporal retorna um dicionário com a DATA como chave em `rates`
    # Exemplo: {"rates": {"2024-01-02": {"BRL": 4.91, ...}, "2024-01-03": {...}}}
    # Precisamos iterar sobre esse dicionário para criar uma linha por (data, moeda)
    rows = []
    for rate_date, rate_dict in json_data["rates"].items():
        for target_currency, rate_value in rate_dict.items():
            rows.append((
                json_data["base"],   # base_currency
                rate_date,           # date string — será convertido na Silver
                json.dumps({         # raw_payload como string JSON
                    "base": json_data["base"],
                    "date": rate_date,
                    "rates": {target_currency: rate_value}
                }),
                batch_id,
                ingestion_ts,
                url_history
            ))

else:
    # Plano B: lê do arquivo local upado para o Volume
    print(f"[PLANO B] Lendo arquivo: {PLAN_B_PATH}")
    raw_text = spark.read.format("text").load(PLAN_B_PATH).collect()[0][0]
    json_data = json.loads(raw_text)

    rows = []
    for rate_date, rate_dict in json_data["rates"].items():
        for target_currency, rate_value in rate_dict.items():
            rows.append((
                json_data["base"],
                rate_date,
                json.dumps({"base": json_data["base"], "date": rate_date, "rates": {target_currency: rate_value}}),
                batch_id,
                ingestion_ts,
                PLAN_B_PATH
            ))

print(f"[INFO] {len(rows)} linhas preparadas para a Bronze.")

In [ ]:
# Célula 9 — Gravar Bronze (append-only)
# Schema explícito da Bronze
bronze_schema = StructType([
    StructField("base_currency",        StringType(),    False),
    StructField("rate_date_str",        StringType(),    False),  # string — parsed na Silver
    StructField("raw_payload",          StringType(),    False),
    StructField("_batch_id",            StringType(),    False),
    StructField("_ingestion_timestamp", TimestampType(), False),
    StructField("_source_url",          StringType(),    False),
])

df_bronze = spark.createDataFrame(rows, schema=bronze_schema)

# Append incremental — nunca sobrescreve dados anteriores
df_bronze.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(TABLE_BRONZE)

print(f"[OK] Bronze gravada. Contagem total:")
spark.sql(f"SELECT COUNT(*) as total_linhas FROM {TABLE_BRONZE}").show()

In [ ]:
# Célula 10 — Ingestão do dia atual (execução diária)
# Esta célula representa a execução diária após a carga histórica inicial
# Execute após a Célula 9 nas próximas execuções do pipeline

if not USE_PLAN_B:
    url_today = f"https://api.frankfurter.app/latest?from={BASE_CURRENCY}&to={TARGET_CURRENCIES}"
    today_data = fetch_frankfurter(url_today)

    if today_data:
        today_batch_id = str(uuid.uuid4())
        today_ts = datetime.utcnow()
        # Uma linha por moeda, cada uma com um único par (target, rate) no payload
        today_rows = [
            (
                today_data["base"],
                today_data["date"],
                json.dumps({"base": today_data["base"], "date": today_data["date"],
                            "rates": {t: r}}),
                today_batch_id,
                today_ts,
                url_today
            )
            for t, r in today_data["rates"].items()
        ]
        df_today = spark.createDataFrame(today_rows, schema=bronze_schema)
        df_today.write.format("delta").mode("append").saveAsTable(TABLE_BRONZE)
        print(f"[OK] Dados de hoje ({today_data['date']}) gravados na Bronze. +{len(today_rows)} linhas.")
    else:
        print("[AVISO] Não foi possível buscar dados de hoje. Bronze mantida como está.")
else:
    print("[PLANO B] Ingestão do dia atual ignorada — usando arquivo local.")

## Passo 2 — Silver: parse, tipagem estrita, Data Quality e MERGE INTO

Princípio Silver: schema fixo, tipos corretos, deduplicação garantida por chave de negócio.
Registros ruins **não são descartados** — são flaggeados com `_dq_status = "invalid"` para auditoria.

**Chave de MERGE:** `(base_currency, target_currency, rate_date)` — garante idempotência.

In [ ]:
# Célula 12 — Parse Bronze → Silver (staging)
from pyspark.sql.types import DoubleType, DateType
from pyspark.sql.functions import col, lit, when, to_date, get_json_object, row_number
from pyspark.sql.window import Window

TABLE_SILVER = f"{CATALOG}.{SCHEMA}.daily_rates"

# Lê a Bronze e faz parse do JSON
df_raw = spark.read.format("delta").table(TABLE_BRONZE)

# Extrai campos do raw_payload via get_json_object
df_parsed = df_raw.select(
    get_json_object(col("raw_payload"), "$.base").alias("base_currency"),
    get_json_object(col("raw_payload"), "$.date").cast(DateType()).alias("rate_date"),
    # O target_currency e exchange_rate estão no único par do rates
    # Extraímos dinamicamente via Python antes — cada linha já tem 1 par
    col("raw_payload"),
    col("_ingestion_timestamp")
)

# Para extrair target_currency e rate de cada linha, usamos o rate_date_str como chave auxiliar
# Cada linha da Bronze já tem um único par (target, rate) — extraímos via UDF simples
from pyspark.sql.functions import udf

@udf(returnType=StringType())
def extract_target(raw_payload: str) -> str:
    try:
        d = json.loads(raw_payload)
        return list(d["rates"].keys())[0]
    except:
        return None

@udf(returnType=DoubleType())
def extract_rate(raw_payload: str) -> float:
    try:
        d = json.loads(raw_payload)
        return float(list(d["rates"].values())[0])
    except:
        return None

df_silver_staging = df_parsed.withColumn("target_currency", extract_target(col("raw_payload"))) \
                              .withColumn("exchange_rate",   extract_rate(col("raw_payload"))) \
                              .drop("raw_payload")

# Deduplicação (window function): a Bronze é append-only, então a mesma chave pode
# aparecer em vários lotes. Mantemos só a ingestão mais recente por chave —
# sem isso o MERGE falha com "multiple source rows matched".
w_dedup = Window.partitionBy("base_currency", "target_currency", "rate_date") \
                .orderBy(col("_ingestion_timestamp").desc())
df_silver_staging = df_silver_staging.withColumn("_rn", row_number().over(w_dedup)) \
                                     .filter(col("_rn") == 1) \
                                     .drop("_rn")

# Data Quality: taxa <= 0 ou None é inválida
df_silver_staging = df_silver_staging.withColumn(
    "_dq_status",
    when(col("exchange_rate").isNull() | (col("exchange_rate") <= 0), lit("invalid"))
    .otherwise(lit("valid"))
)

print("[INFO] Schema da Silver staging:")
df_silver_staging.printSchema()
print(f"\n[INFO] Distribuição DQ:")
df_silver_staging.groupBy("_dq_status").count().show()

In [ ]:
# Célula 13 — Criar tabela Silver e executar MERGE INTO
# Cria a tabela Silver se não existir
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_SILVER} (
    base_currency        STRING    NOT NULL,
    target_currency      STRING    NOT NULL,
    rate_date            DATE      NOT NULL,
    exchange_rate        DOUBLE,
    _dq_status           STRING,
    _ingestion_timestamp TIMESTAMP
)
USING DELTA
COMMENT 'Taxas de câmbio diárias normalizadas — Frankfurter API'
""")

# Registra o DataFrame de staging como view temporária para o MERGE
df_silver_staging.createOrReplaceTempView("silver_staging")

# MERGE INTO: idempotente por chave composta (base, target, date)
spark.sql(f"""
MERGE INTO {TABLE_SILVER} AS target
USING silver_staging AS source
ON  target.base_currency   = source.base_currency
AND target.target_currency = source.target_currency
AND target.rate_date       = source.rate_date
WHEN MATCHED THEN
    UPDATE SET
        exchange_rate        = source.exchange_rate,
        _dq_status           = source._dq_status,
        _ingestion_timestamp = source._ingestion_timestamp
WHEN NOT MATCHED THEN
    INSERT (base_currency, target_currency, rate_date, exchange_rate, _dq_status, _ingestion_timestamp)
    VALUES (source.base_currency, source.target_currency, source.rate_date,
            source.exchange_rate, source._dq_status, source._ingestion_timestamp)
""")

print("[OK] MERGE INTO executado.")
spark.sql(f"SELECT COUNT(*) as total_silver FROM {TABLE_SILVER}").show()
spark.sql(f"SELECT COUNT(*) as registros_2024 FROM {TABLE_SILVER} WHERE rate_date BETWEEN '2024-01-01' AND '2024-12-31'").show()

## Passo 3 — Gold: resumo mensal USD/BRL + OPTIMIZE

Princípio Gold: dados prontos para consumo. Nenhuma lógica de negócio nos dashboards — ela fica aqui.

Calculamos por mês-ano: taxa mínima, máxima, média e desvio padrão do par USD→BRL.

In [ ]:
# Célula 15 — Construir tabela Gold
from pyspark.sql.functions import date_format, avg, min as spark_min, max as spark_max, stddev

TABLE_GOLD = f"{CATALOG}.{SCHEMA}.usd_brl_monthly_summary"

df_gold = spark.sql(f"""
    SELECT
        date_format(rate_date, 'yyyy-MM') AS year_month,
        COUNT(*)                          AS trading_days,
        ROUND(AVG(exchange_rate), 4)      AS avg_rate,
        ROUND(MIN(exchange_rate), 4)      AS min_rate,
        ROUND(MAX(exchange_rate), 4)      AS max_rate,
        ROUND(STDDEV(exchange_rate), 4)   AS stddev_rate
    FROM {TABLE_SILVER}
    WHERE base_currency   = 'USD'
      AND target_currency = 'BRL'
      AND _dq_status      = 'valid'
      AND rate_date BETWEEN '2024-01-01' AND '2024-12-31'
    GROUP BY date_format(rate_date, 'yyyy-MM')
    ORDER BY year_month
""")

df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLE_GOLD)

print(f"[OK] Tabela Gold gravada: {TABLE_GOLD}")
display(spark.table(TABLE_GOLD))

In [ ]:
%sql
-- Célula 16 — OPTIMIZE + ZORDER na Gold
OPTIMIZE workshop_weather.exchange.usd_brl_monthly_summary
ZORDER BY (year_month);

-- Valida o resultado da otimização
DESCRIBE DETAIL workshop_weather.exchange.usd_brl_monthly_summary;

In [ ]:
# Célula 17 — Validação final e perguntas de fixação
print("=" * 60)
print("VALIDAÇÃO FINAL — Desafio 1")
print("=" * 60)

# Conta linhas por camada
bronze_count = spark.sql(f"SELECT COUNT(*) as n FROM {TABLE_BRONZE}").collect()[0][0]
silver_count = spark.sql(f"SELECT COUNT(*) as n FROM {TABLE_SILVER}").collect()[0][0]
gold_count   = spark.sql(f"SELECT COUNT(*) as n FROM {TABLE_GOLD}").collect()[0][0]

print(f"Bronze (raw_frankfurter):       {bronze_count:,} linhas")
print(f"Silver (daily_rates):           {silver_count:,} linhas")
print(f"Gold   (usd_brl_monthly_summary): {gold_count} linhas (esperado: 12 para 2024)")

# Mês com maior variação
spark.sql(f"SELECT year_month, stddev_rate FROM {TABLE_GOLD} ORDER BY stddev_rate DESC LIMIT 1") \
     .show()

print("""
PERGUNTAS DE FIXAÇÃO (responda mentalmente):
1. Por que a Bronze usa mode("append") mas a Silver usa MERGE INTO?
2. O que acontece se você executar o notebook Silver 3 vezes seguidas?
   O COUNT(*) da Silver deve mudar? Por quê?
3. Por que a série temporal da Frankfurter exige um loop Python antes
   do createDataFrame, diferente do endpoint /latest?
""")